<a href="https://colab.research.google.com/github/huiningjiao02-ship-it/Huining-JIAO/blob/main/%E8%AE%BA%E6%96%87%E5%A4%8D%E7%8E%B0EEG_Conformer_Convolutional_Transformer_for_EEG_Decoding_and_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title EEG Conformer 复现 | PhysioNet eegmmidb 数据集版
# @markdown 🔹 自动下载并适配 PhysioNet EEG 运动想象数据集
# @markdown 🔹 快速验证：默认训练前3个被试，200轮
# @markdown 🔹 完整复现：修改下方参数 n_subjects=109, n_epochs=2000

# ===================== 可修改参数 =====================
n_subjects = 7   # 训练被试数量 1-109，完整复现设为109
n_epochs = 500   # 训练轮数，论文原版2000，快速验证设为200
batch_size = 32  # eegmmidb单被试数据量少，调小batch防止显存不足
learning_rate = 0.0002
# ======================================================

# 1. 安装核心依赖 (新增 mne 用于读取 EDF 文件)
!pip install -q torch torchvision torchaudio numpy scipy matplotlib scikit-learn mne einops

# 2. 环境与路径初始化
import os
import sys
import time
import datetime
import random
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.autograd import Variable
from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce
from torch.backends import cudnn
import mne

# 固定随机种子，保证结果可复现
seed_n = np.random.randint(2021)
random.seed(seed_n)
np.random.seed(seed_n)
torch.manual_seed(seed_n)
torch.cuda.manual_seed(seed_n)
torch.cuda.manual_seed_all(seed_n)
cudnn.benchmark = False
cudnn.deterministic = True

# 路径配置（完全适配Colab）
os.chdir('/content')
project_path = '/content/EEG-Conformer'
data_path = '/content/physionet.org/files/eegmmidb/1.0.0/'  # eegmmidb下载路径
result_path = project_path + '/results_eegmmidb/'

# 自动补全目录
if not os.path.exists(project_path):
    !git clone https://github.com/eeyhsong/EEG-Conformer.git
os.chdir(project_path)
os.makedirs(result_path, exist_ok=True)

# ===================== 3. 自动下载 PhysioNet eegmmidb 数据集 =====================
print("正在检查/下载 PhysioNet eegmmidb 数据集...")
# 【修改开始】注释掉完整下载，只下载前10个被试的核心运动想象文件
if not os.path.exists(data_path) or len(os.listdir(data_path)) < 10:
    # !wget -r -N -c -np https://physionet.org/files/eegmmidb/1.0.0/ # 【注释掉】这行全量下载

    # 【启用】快速演示版：只下载前10个被试的运动想象文件 (R11-R14)
    print("开始快速下载：前10个被试的核心运动想象数据...")
    for sub in range(1, 11): # 只下载 S001 到 S010
        sub_str = f'S{sub:03d}'
        sub_path = os.path.join(data_path, sub_str)
        os.makedirs(sub_path, exist_ok=True)
        # 只下载运动想象的 Run (R11-R14)，数据量小且是核心任务
        for run in [11, 12, 13, 14]:
            filename = f'{sub_str}R{run:02d}.edf'
            file_url = f'https://physionet.org/files/eegmmidb/1.0.0/{sub_str}/{filename}'
            if not os.path.exists(os.path.join(sub_path, filename)):
                !wget -q -P {sub_path} {file_url}
                print(f"  已下载: {filename}")
    print("快速下载完成！")
else:
    print("数据集已存在，跳过下载。")
# 【修改结束】
# GPU配置
gpus = [0]
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ["CUDA_VISIBLE_DEVICES"] = ','.join(map(str, gpus))
Tensor = torch.cuda.FloatTensor
LongTensor = torch.cuda.LongTensor

# ===================== 4. 修改后的模型定义 (适配64通道) =====================
class PatchEmbedding(nn.Module):
    def __init__(self, emb_size=40):
        super().__init__()
        self.shallownet = nn.Sequential(
            nn.Conv2d(1, 40, (1, 25), (1, 1)),
            # 【核心修改】这里从 (22, 1) 改为 (64, 1)，适配 eegmmidb 的64个EEG通道
            nn.Conv2d(40, 40, (64, 1), (1, 1)),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.AvgPool2d((1, 75), (1, 15)),
            nn.Dropout(0.5),
        )
        self.projection = nn.Sequential(
            nn.Conv2d(40, emb_size, (1, 1), stride=(1, 1)),
            Rearrange('b e (h) (w) -> b (h w) e'),
        )

    def forward(self, x):
        x = self.shallownet(x)
        x = self.projection(x)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.keys = nn.Linear(emb_size, emb_size)
        self.queries = nn.Linear(emb_size, emb_size)
        self.values = nn.Linear(emb_size, emb_size)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x, mask=None):
        queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
        keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
        values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
        energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy.mask_fill(~mask, fill_value)
        scaling = self.emb_size ** (1 / 2)
        att = F.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum('bhal, bhlv -> bhav ', att, values)
        out = rearrange(out, "b h n d -> b n (h d)")
        out = self.projection(out)
        return out

class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x

class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion, drop_p):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
        )

class TransformerEncoderBlock(nn.Sequential):
    def __init__(self, emb_size, num_heads=10, drop_p=0.5, forward_expansion=4, forward_drop_p=0.5):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            ))
        )

class TransformerEncoder(nn.Sequential):
    def __init__(self, depth, emb_size):
        super().__init__(*[TransformerEncoderBlock(emb_size) for _ in range(depth)])

class ClassificationHead(nn.Sequential):
    def __init__(self, emb_size, n_classes):
        super().__init__()
        # 【注意】由于通道数变了，这里的输入维度也会变，需要根据池化后的输出调整
        # 64通道下，池化后维度计算为 40 * 61 = 2440 (和之前一样，因为池化是在时间维度)
        self.fc = nn.Sequential(
            nn.Linear(2440, 256),
            nn.ELU(),
            nn.Dropout(0.5),
            nn.Linear(256, 32),
            nn.ELU(),
            nn.Dropout(0.3),
            nn.Linear(32, 4) # 保持4分类：左手/右手/脚/舌头
        )

    def forward(self, x):
        x = x.contiguous().view(x.size(0), -1)
        out = self.fc(x)
        return x, out

class Conformer(nn.Sequential):
    def __init__(self, emb_size=40, depth=6, n_classes=4, **kwargs):
        super().__init__(
            PatchEmbedding(emb_size),
            TransformerEncoder(depth, emb_size),
            ClassificationHead(emb_size, n_classes)
        )

# ===================== 5. 新增：PhysioNet eegmmidb 数据读取与预处理函数 =====================
def load_eegmmidb_subject(sub_id, base_path):
    """
    加载单个被试的 eegmmidb 数据
    任务映射：T1=左手(0), T2=右手(1), T3=脚(2), T4=舌头(3)
    """
    sfreq = 160  # eegmmidb 原始采样率是 160 Hz
    target_sfreq = 250 # 重采样到和原论文一致的 250 Hz
    t_start = 0.5  # cue后0.5s开始
    t_end = 4.5    # cue后4.5s结束，共4s

    # eegmmidb 的任务编号映射 (Run编号 -> 任务类型)
    # 我们只用运动想象的 Run (R11-R14)，如果数据不够可以加上运动执行 (R03-R06)
    runs = [11, 12, 13, 14] # 运动想象
    # runs = [3,4,5,6,11,12,13,14] # 运动想象+执行

    sub_str = f'S{sub_id:03d}'
    sub_path = os.path.join(base_path, sub_str)

    all_trials = []
    all_labels = []

    for run in runs:
        edf_file = os.path.join(sub_path, f'{sub_str}R{run:02d}.edf')
        if not os.path.exists(edf_file):
            continue

        # 使用 mne 读取 EDF 文件
        raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)
        # 只保留 EEG 通道 (剔除状态通道)
        eeg_channels = [ch for ch in raw.ch_names if 'EEG' in ch]
        raw.pick_channels(eeg_channels)
        # 重采样到 250 Hz
        raw.resample(target_sfreq, verbose=False)

        # 提取事件 (从 annotations 中)
        events, event_id = mne.events_from_annotations(raw, verbose=False)

        # 事件映射：T1/T2 根据不同Run代表不同任务
        # R03/R11: T1=左手, T2=右手
        # R04/R12: T1=左手, T2=舌头
        # R05/R13: T1=脚, T2=舌头
        # R06/R14: T1=脚, T2=右手
        # 为了简化，我们统一映射为 4类：
        # 左手=0, 右手=1, 脚=2, 舌头=3
        task_map = {
            3: { 'T1': 0, 'T2': 1 }, # R03: 左 vs 右
            4: { 'T1': 0, 'T2': 3 }, # R04: 左 vs 舌
            5: { 'T1': 2, 'T2': 3 }, # R05: 脚 vs 舌
            6: { 'T1': 2, 'T2': 1 }, # R06: 脚 vs 右
            11: { 'T1': 0, 'T2': 1 }, # R11: 左 vs 右 (想象)
            12: { 'T1': 0, 'T2': 3 }, # R12: 左 vs 舌 (想象)
            13: { 'T1': 2, 'T2': 3 }, # R13: 脚 vs 舌 (想象)
            14: { 'T1': 2, 'T2': 1 }, # R14: 脚 vs 右 (想象)
        }

        # 切分 trial
        data = raw.get_data() * 1e6 # 转换为 uV
        sfreq_new = raw.info['sfreq']
        start_idx = int(t_start * sfreq_new)
        end_idx = int(t_end * sfreq_new)
        n_samples = end_idx - start_idx

        for event in events:
            onset = event[0]
            trigger = event[2]
            trigger_str = list(event_id.keys())[list(event_id.values()).index(trigger)]

            if trigger_str not in ['T1', 'T2']:
                continue

            # 获取标签
            if run not in task_map or trigger_str not in task_map[run]:
                continue
            label = task_map[run][trigger_str]

            # 截取数据
            trial_start = onset + start_idx
            trial_end = onset + end_idx
            if trial_end > data.shape[1]:
                continue

            trial_data = data[:, trial_start:trial_end]

            # 确保长度一致 (补零或截断)
            if trial_data.shape[1] < n_samples:
                pad_len = n_samples - trial_data.shape[1]
                trial_data = np.pad(trial_data, ((0,0),(0,pad_len)), mode='constant')
            else:
                trial_data = trial_data[:, :n_samples]

            # 只取前64个通道 (防止有些文件通道数不一致)
            if trial_data.shape[0] > 64:
                trial_data = trial_data[:64, :]

            all_trials.append(trial_data)
            all_labels.append(label)

    if len(all_trials) == 0:
        return None, None

    # 转换为模型要求的格式: [n_trials, 1, n_channels, n_times]
    data = np.stack(all_trials, axis=0)
    data = np.expand_dims(data, axis=1)
    labels = np.array(all_labels)

    return data, labels

# ===================== 6. 训练流程类 (适配 eegmmidb) =====================
class ExP():
    def __init__(self, nsub):
        super(ExP, self).__init__()
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.c_dim = 4
        self.lr = learning_rate
        self.b1 = 0.5
        self.b2 = 0.999
        self.nSub = nsub
        self.root = data_path

        self.log_write = open(f"{result_path}/log_subject{self.nSub}.txt", "w")
        self.criterion_cls = torch.nn.CrossEntropyLoss().cuda()
        self.model = Conformer().cuda()
        self.model = nn.DataParallel(self.model, device_ids=[i for i in range(len(gpus))])
        self.model = self.model.cuda()

    # 论文原版数据增强方法 (已适配64通道)
    def interaug(self, timg, label):
        aug_data = []
        aug_label = []
        for cls4aug in range(4):
            cls_idx = np.where(label == cls4aug + 1) # 注意这里label是1-based用于增强
            tmp_data = timg[cls_idx]
            tmp_label = label[cls_idx]
            if len(tmp_data) == 0:
                continue
            tmp_aug_data = np.zeros((int(self.batch_size / 4), 1, 64, 1000)) # 【修改】64通道
            for ri in range(int(self.batch_size / 4)):
                for rj in range(8):
                    rand_idx = np.random.randint(0, tmp_data.shape[0], 8)
                    tmp_aug_data[ri, :, :, rj * 125:(rj + 1) * 125] = tmp_data[rand_idx[rj], :, :, rj * 125:(rj + 1) * 125]
            aug_data.append(tmp_aug_data)
            aug_label.append(tmp_label[:int(self.batch_size / 4)])
        if len(aug_data) == 0:
            return None, None
        aug_data = np.concatenate(aug_data)
        aug_label = np.concatenate(aug_label)
        aug_shuffle = np.random.permutation(len(aug_data))
        aug_data = aug_data[aug_shuffle, :, :]
        aug_label = aug_label[aug_shuffle]
        aug_data = torch.from_numpy(aug_data).cuda()
        aug_data = aug_data.float()
        aug_label = torch.from_numpy(aug_label-1).cuda() # 转回0-based
        aug_label = aug_label.long()
        return aug_data, aug_label

    # 适配后的数据加载
    def get_source_data(self):
        # 加载 eegmmidb 数据
        self.allData, self.allLabel = load_eegmmidb_subject(self.nSub, self.root)
        if self.allData is None:
            return None, None, None, None

        # eegmmidb没有官方划分Train/Test，我们手动 8:2 划分
        n_trials = len(self.allData)
        n_train = int(0.8 * n_trials)

        # 打乱
        shuffle_num = np.random.permutation(n_trials)
        self.allData = self.allData[shuffle_num, :, :, :]
        self.allLabel = self.allLabel[shuffle_num]

        # 划分
        train_data = self.allData[:n_train]
        train_label = self.allLabel[:n_train]
        test_data = self.allData[n_train:]
        test_label = self.allLabel[n_train:]

        # 全局标准化 (仅用训练集统计量)
        target_mean = np.mean(train_data)
        target_std = np.std(train_data)
        self.allData = (self.allData - target_mean) / target_std
        train_data = (train_data - target_mean) / target_std
        test_data = (test_data - target_mean) / target_std

        return train_data, train_label+1, test_data, test_label+1 # label+1是为了适配增强函数

    # 训练主循环
    def train(self):
        img, label, test_data, test_label = self.get_source_data()
        if img is None:
            print(f"被试 {self.nSub} 数据加载失败，跳过。")
            return 0, 0, 0, 0

        img = torch.from_numpy(img)
        label = torch.from_numpy(label - 1) # 转回 0-based
        dataset = TensorDataset(img, label)
        self.dataloader = DataLoader(dataset=dataset, batch_size=self.batch_size, shuffle=True)

        test_data = torch.from_numpy(test_data)
        test_label = torch.from_numpy(test_label - 1)
        test_dataset = TensorDataset(test_data, test_label)
        self.test_dataloader = DataLoader(dataset=test_dataset, batch_size=self.batch_size, shuffle=True)

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, betas=(self.b1, self.b2))
        test_data = Variable(test_data.type(Tensor))
        test_label = Variable(test_label.type(LongTensor))

        bestAcc = 0
        averAcc = 0
        num = 0
        Y_true = 0
        Y_pred = 0

        print(f'===== 开始训练被试 {self.nSub} | 样本数: {len(img)} =====')
        for e in range(self.n_epochs):
            self.model.train()
            for i, (img_batch, label_batch) in enumerate(self.dataloader):
                img_batch = Variable(img_batch.cuda().type(Tensor))
                label_batch = Variable(label_batch.cuda().type(LongTensor))

                # 数据增强
                aug_data, aug_label = self.interaug(self.allData, self.allLabel+1)
                if aug_data is not None:
                    img_batch = torch.cat((img_batch, aug_data))
                    label_batch = torch.cat((label_batch, aug_label))

                tok, outputs = self.model(img_batch)
                loss = self.criterion_cls(outputs, label_batch)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

            # 每轮测试
            self.model.eval()
            with torch.no_grad():
                Tok, Cls = self.model(test_data)
                loss_test = self.criterion_cls(Cls, test_label)
                y_pred = torch.max(Cls, 1)[1]
                acc = float((y_pred == test_label).cpu().numpy().astype(int).sum()) / float(test_label.size(0))
                train_pred = torch.max(outputs, 1)[1]
                train_acc = float((train_pred == label_batch).cpu().numpy().astype(int).sum()) / float(label_batch.size(0))

                if (e+1) % 20 == 0: # eegmmidb训练快，每20轮打印一次
                    print(f'Epoch: {e+1:4d} | Train loss: {loss.detach().cpu().numpy():.6f} | Test acc: {acc:.4f}')
                self.log_write.write(str(e) + "    " + str(acc) + "\n")

                num += 1
                averAcc += acc
                if acc > bestAcc:
                    bestAcc = acc
                    Y_true = test_label
                    Y_pred = y_pred

        torch.save(self.model.module.state_dict(), f'{result_path}/model_sub{self.nSub}.pth')
        averAcc = averAcc / num
        print(f'===== 被试 {self.nSub} 训练完成 | 最高准确率: {bestAcc:.4f} =====\n')
        self.log_write.write('The average accuracy is: ' + str(averAcc) + "\n")
        self.log_write.write('The best accuracy is: ' + str(bestAcc) + "\n")
        self.log_write.close()
        return bestAcc, averAcc, Y_true, Y_pred

# ===================== 主函数 =====================
def main():
    best_total = 0
    aver_total = 0
    valid_count = 0
    result_write = open(f"{result_path}/all_subjects_result.txt", "w")
    print(f'===== EEG Conformer @ PhysioNet eegmmidb 训练开始 =====\n')

    for i in range(1, n_subjects+1):
        starttime = datetime.datetime.now()
        sub_id = i
        exp = ExP(sub_id)
        bestAcc, averAcc, Y_true, Y_pred = exp.train()

        if bestAcc > 0:
            result_write.write(f'Subject {sub_id} | Best accuracy: {bestAcc:.6f}\n')
            result_write.write(f'Subject {sub_id} | Average accuracy: {averAcc:.6f}\n\n')
            best_total += bestAcc
            aver_total += averAcc
            valid_count += 1

            if valid_count == 1:
                yt_all = Y_true
                yp_all = Y_pred
            elif Y_true is not None:
                yt_all = torch.cat((yt_all, Y_true))
                yp_all = torch.cat((yp_all, Y_pred))

        endtime = datetime.datetime.now()

    if valid_count > 0:
        best_total = best_total / valid_count
        aver_total = aver_total / valid_count
        print(f'===== 全部训练完成 | 有效被试数: {valid_count} =====')
        print(f'平均最高准确率: {best_total:.6f}')
        result_write.write(f'**平均最高准确率: {best_total:.6f}\n')
        result_write.close()

    print(f'\n所有日志和模型权重已保存到: {result_path}')

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 114.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.
Cloning into 'EEG-Conformer'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 149 (delta 1), reused 1 (delta 1), pack-reused 146 (from 3)
Receiving objects: 100% (149/149), 3.53 MiB | 18.08 MiB/s, done.
Resolving deltas: 100% (65/65), done.
正在检查/下载 PhysioNet eegmmidb 数据集...
开始快速下载：前10个被试的核心运动想象数据...
  已下载: S001R11.edf
  已下载: S001R12.edf
  已下载: S001R13.edf
  已下载: S001R14.edf
  已下载: S002R11.edf
  已下载: S002R12.edf
  已下载: S002R13.edf
  已下载: S002R14.edf
  已下载: S003R11.edf
  已下载: S003R12.edf